In [18]:
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets
import torchvision.transforms.v2 as transforms  # Nueva versión de transforms
import matplotlib.pyplot as plt

import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.is_available()

False

In [19]:
!apt-get install -y p7zip-full
!wget https://github.com/ichaparroc/IA-EPIS/raw/refs/heads/main/FDL_2.7z -O FDL_2.7z
!7z x FDL_2.7z

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
p7zip-full is already the newest version (16.02+dfsg-8).
0 upgraded, 0 newly installed, 0 to remove and 34 not upgraded.
--2025-05-01 18:08:34--  https://github.com/ichaparroc/IA-EPIS/raw/refs/heads/main/FDL_2.7z
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/ichaparroc/IA-EPIS/refs/heads/main/FDL_2.7z [following]
--2025-05-01 18:08:34--  https://raw.githubusercontent.com/ichaparroc/IA-EPIS/refs/heads/main/FDL_2.7z
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 24287051 (23M) [application/octet-str

In [20]:
IMG_HEIGHT = 28
IMG_WIDTH = 28
IMG_CHS = 1
N_CLASSES = 24

train_df = pd.read_csv("sign_mnist_train.csv")
valid_df = pd.read_csv("sign_mnist_valid.csv")

class MyDataset(Dataset):
    def __init__(self, base_df):
        x_df = base_df.copy()
        y_df = x_df.pop('label')
        x_df = x_df.values / 255  # Normalize values from 0 to 1
        x_df = x_df.reshape(-1, IMG_CHS, IMG_WIDTH, IMG_HEIGHT)
        self.xs = torch.tensor(x_df).float().to(device)
        self.ys = torch.tensor(y_df).to(device)

    def __getitem__(self, idx):
        x = self.xs[idx]
        y = self.ys[idx]
        return x, y

    def __len__(self):
        return len(self.xs)

n = 32
train_data = MyDataset(train_df)
train_loader = DataLoader(train_data, batch_size=n, shuffle=True)
train_N = len(train_loader.dataset)

valid_data = MyDataset(valid_df)
valid_loader = DataLoader(valid_data, batch_size=n)
valid_N = len(valid_loader.dataset)

In [21]:
class MyConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout_p):
        kernel_size = 3
        super().__init__()

        self.model = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size, stride=1, padding=1),
            nn.BatchNorm2d(out_ch), #a veces no conviene
            nn.ReLU(), #funcion de activacion
            nn.Dropout(dropout_p), # reduce el sobreajuste
            nn.MaxPool2d(2, stride=2) #achicamos
        )

    def forward(self, x):
        return self.model(x)

In [22]:
flattened_img_size = 75 * 3 * 3

# Input 1 x 28 x 28
base_model = nn.Sequential(
    MyConvBlock(IMG_CHS, 25, 0), # 25 x 14 x 14
    MyConvBlock(25, 50, 0.2), # 50 x 7 x 7
    MyConvBlock(50, 75, 0),  # 75 x 3 x 3
    # Flatten to Dense Layers
    nn.Flatten(),
    nn.Linear(flattened_img_size, 512),
    nn.Dropout(.3),
    nn.ReLU(),
    nn.Linear(512, N_CLASSES)
)

In [23]:
loss_function = nn.CrossEntropyLoss()
optimizer = Adam(base_model.parameters())

model = base_model.to(device)

In [24]:
def get_batch_accuracy(output, y, N):
    pred = output.argmax(dim=1, keepdim=True)
    correct = pred.eq(y.view_as(pred)).sum().item()
    return correct / N

In [25]:
def train():
    loss = 0
    accuracy = 0

    model.train()
    for x, y in train_loader:
        output = model(x)  # Updated
        optimizer.zero_grad()
        batch_loss = loss_function(output, y)
        batch_loss.backward()
        optimizer.step()

        loss += loss_function(output, y).item()
        accuracy += get_batch_accuracy(output, y, train_N)
    print('Train - Loss: {:.4f} Accuracy: {:.4f}'.format(loss, accuracy))

In [26]:
def validate():
    loss = 0
    accuracy = 0

    model.eval()
    with torch.no_grad():
        for x, y in valid_loader:
            output = model(x)

            loss += loss_function(output, y).item()
            accuracy += get_batch_accuracy(output, y, valid_N)
    print('Valid - Loss: {:.4f} Accuracy: {:.4f}'.format(loss, accuracy))

In [27]:
import torch._dynamo
torch._dynamo.config.suppress_errors = True

epochs = 6

for epoch in range(epochs):
    print('Epoch: {}'.format(epoch))
    train()
    validate()

Epoch: 0
Train - Loss: 271.0839 Accuracy: 0.9070
Valid - Loss: 29.1122 Accuracy: 0.9591
Epoch: 1
Train - Loss: 17.6479 Accuracy: 0.9954
Valid - Loss: 32.9647 Accuracy: 0.9402
Epoch: 2
Train - Loss: 11.5747 Accuracy: 0.9962
Valid - Loss: 34.8820 Accuracy: 0.9476
Epoch: 3
Train - Loss: 6.5241 Accuracy: 0.9979
Valid - Loss: 54.6014 Accuracy: 0.9251
Epoch: 4
Train - Loss: 10.5075 Accuracy: 0.9961
Valid - Loss: 13.3029 Accuracy: 0.9785
Epoch: 5
Train - Loss: 0.2381 Accuracy: 1.0000
Valid - Loss: 14.6115 Accuracy: 0.9766


Para determinar la mejor época, debemos equilibrar dos cosas:

✅ Alta precisión (accuracy) en validación.

📉 Bajo valor de pérdida (loss) en validación, ya que refleja cuán seguras y confiables son las predicciones.

| Época | Valid Accuracy | Valid Loss |
|-------|----------------|------------|
| 0     | 0.9711         | 18.8149    |
| 1     | 0.9756         | 16.1283    |
| 2     | 0.9741         | 16.4520    |
| 3     | 0.9717         | 21.8534    |
| 4     | 0.9601         | 41.8180    |
| 5     | 0.9759         | 22.0275    |
| 6     | 0.9796 ✅       | 15.4920 ✅ |
| 7     | 0.9465         | 46.4901    |
| 8     | 0.9766         | 17.4601    |
| 9     | 0.9798 ✅       | 15.6153    |

  Otra forma de determinar la mejor época es una balanceo entre la precision y pérdida: Score=Validation Accuracy−λ×Validation Loss

| Epoch | Accuracy | Loss    | Score                  |
|-------|----------|---------|-------------------------|
| 0     | 0.9711   | 18.8149 | 0.9711 - 0.1881 = **0.7830** |
| 1     | 0.9756   | 16.1283 | 0.9756 - 0.1613 = **0.8143** |
| 2     | 0.9741   | 16.4520 | 0.9741 - 0.1645 = **0.8096** |
| 3     | 0.9717   | 21.8534 | 0.9717 - 0.2185 = **0.7532** |
| 4     | 0.9601   | 41.8180 | 0.9601 - 0.4182 = **0.5419** |
| 5     | 0.9759   | 22.0275 | 0.9759 - 0.2203 = **0.7556** |
| 6     | 0.9796   | 15.4920 | 0.9796 - 0.1549 = **0.8247** ✅ |
| 7     | 0.9465   | 46.4901 | 0.9465 - 0.4649 = **0.4816** |
| 8     | 0.9766   | 17.4601 | 0.9766 - 0.1746 = **0.8020** |
| 9     | 0.9798   | 15.6153 | 0.9798 - 0.1561 = **0.8237** |

In [28]:
torch.save(base_model, 'model.pth')